# LeetCode #129 — Sum Root to Leaf Numbers: DFS mang theo số đang ghép

**Mức độ:** Medium · **Chủ đề:** Tree, Depth-First Search, Binary Tree

Skeleton trên LeetCode:

```python
class Solution:
    def sumNumbers(self, root: Optional[TreeNode]) -> int:
```

Mục tiêu của notebook:

1. Ghép chữ số khi đi xuống: `cur = cur * 10 + node.val` (giống đọc số từ trái sang phải).
2. DFS truyền **giá trị** (không cần backtracking vì số nguyên bất biến).
3. Ba cách cài đặt: đệ quy, stack, BFS — và cách `O(1)` bộ nhớ bằng Morris (tham khảo).
4. Điều kiện lá, và vì sao không cộng ở nút `None`.

# 1. Bài toán là gì?

Cho `root` của cây nhị phân, mỗi nút chứa **một chữ số** `0..9`. Mỗi đường từ gốc tới lá biểu diễn một **số**: ví dụ đường `1 → 2 → 3` là số `123`. Trả về **tổng** các số của mọi đường gốc → lá.

**Constraints (theo đề):**

- Số nút trong khoảng `[1, 1000]`.
- `0 <= Node.val <= 9`
- Độ sâu cây không quá `10`.
- Đáp án vừa số nguyên 32-bit.

---

## Ví dụ

```text
    1
   / \          ->  25     (12 + 13)
  2   3

      4
     / \
    9   0       ->  1026   (495 + 491 + 40)
   / \
  5   1
```

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Cây nhị phân (binary tree)
2. Duyệt theo chiều sâu (DFS — Depth-First Search)

Cuối mục có phần **2.3 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Cây nhị phân (binary tree)

### Cây nhị phân là gì?

Cây nhị phân gồm các **nút**; mỗi nút có một giá trị và tối đa **hai con**: con **trái** và con **phải**. Nút trên cùng là **gốc** (root); nút không có con là **lá** (leaf).

```text
          1          ← gốc (độ sâu 0)
        /   \
       2     3       ← độ sâu 1
      / \     \
     4   5     6     ← lá 4, 5, 6 (độ sâu 2)
```

Thuật ngữ: **cha/con**, **anh em** (cùng cha), **cây con** (một nút cùng toàn bộ con cháu), **độ sâu** (số cạnh từ gốc), **chiều cao** (số tầng / đường dài nhất từ gốc tới lá).

In [1]:
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

# Dựng cây ở hình trên bằng tay
goc = TreeNode(1,
               TreeNode(2, TreeNode(4), TreeNode(5)),
               TreeNode(3, None, TreeNode(6)))
print("gốc:", goc.val, "| con trái:", goc.left.val, "| con phải:", goc.right.val,
      "| 3 có con trái?", goc.right.left)

gốc: 1 | con trái: 2 | con phải: 3 | 3 có con trái? None


### Biểu diễn kiểu LeetCode: danh sách theo tầng (level-order)

LeetCode ghi cây thành danh sách đọc từng tầng, từ trái sang phải, `None` cho chỗ trống: cây trên là `[1, 2, 3, 4, 5, None, 6]`. Dựng cây từ danh sách này bằng một **hàng đợi**: lần lượt lấy nút cha ra, gán hai giá trị kế tiếp làm con trái/phải.

In [2]:
from collections import deque

def tao_cay(values):
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    q, i = deque([root]), 1
    while q and i < len(values):
        node = q.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i]); q.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i]); q.append(node.right)
        i += 1
    return root

def ve_cay(node, pre="", la_phai=True, goc_cay=True):
    """In cây nằm ngang để dễ nhìn."""
    if node is None:
        return
    ve_cay(node.right, pre + ("" if goc_cay else ("    " if la_phai else "│   ")), True, False)
    print(pre + ("" if goc_cay else ("┌── " if la_phai else "└── ")) + str(node.val))
    ve_cay(node.left, pre + ("" if goc_cay else ("│   " if la_phai else "    ")), False, False)

ve_cay(tao_cay([1, 2, 3, 4, 5, None, 6]))

    ┌── 6
┌── 3
1
│   ┌── 5
└── 2
    └── 4


(Cây in nằm ngang: gốc ở bên trái, con **phải** ở phía **trên**, con **trái** ở phía **dưới**.)

### Ba cách duyệt theo chiều sâu

Với mỗi nút ta làm ba việc: thăm **nút**, duyệt cây con **trái**, duyệt cây con **phải**. Thứ tự của ba việc cho ba kiểu duyệt:

| Kiểu | Thứ tự | Cây trên |
|---|---|---|
| **Preorder** (tiền thứ tự) | nút → trái → phải | `1 2 4 5 3 6` |
| **Inorder** (trung thứ tự) | trái → nút → phải | `4 2 5 1 3 6` |
| **Postorder** (hậu thứ tự) | trái → phải → nút | `4 5 2 6 3 1` |

Mẹo nhớ: "pre/in/post" chỉ **vị trí của nút** so với hai cây con.

In [3]:
def preorder(node, out):
    if node is None:
        return
    out.append(node.val)        # nút
    preorder(node.left, out)    # trái
    preorder(node.right, out)   # phải

def inorder(node, out):
    if node is None:
        return
    inorder(node.left, out)
    out.append(node.val)
    inorder(node.right, out)

def postorder(node, out):
    if node is None:
        return
    postorder(node.left, out)
    postorder(node.right, out)
    out.append(node.val)

cay = tao_cay([1, 2, 3, 4, 5, None, 6])
for ten, f in [("preorder", preorder), ("inorder", inorder), ("postorder", postorder)]:
    out = []; f(cay, out); print(f"{ten:<9}:", out)

preorder : [1, 2, 4, 5, 3, 6]
inorder  : [4, 2, 5, 1, 3, 6]
postorder: [4, 5, 2, 6, 3, 1]


### Duyệt không đệ quy bằng ngăn xếp

Đệ quy thực chất dùng *call stack*; ta có thể tự quản lý ngăn xếp. Preorder: lấy nút ra, ghi lại, đẩy con **phải trước** rồi con **trái** (để trái được lấy ra trước). Inorder: đi trái hết mức, lấy ra, rồi sang phải.

In [4]:
def preorder_lap(root):
    out, st = [], [root] if root else []
    while st:
        node = st.pop()
        out.append(node.val)
        if node.right: st.append(node.right)
        if node.left:  st.append(node.left)
    return out

def inorder_lap(root):
    out, st, cur = [], [], root
    while cur or st:
        while cur:              # đi trái hết mức
            st.append(cur); cur = cur.left
        cur = st.pop()
        out.append(cur.val)
        cur = cur.right
    return out

print("preorder lặp:", preorder_lap(cay), "| inorder lặp:", inorder_lap(cay))

preorder lặp: [1, 2, 4, 5, 3, 6] | inorder lặp: [4, 2, 5, 1, 3, 6]


### Tư duy đệ quy trên cây

Hầu hết bài về cây giải bằng cách hỏi: "nếu đã biết đáp án cho **cây con trái** và **cây con phải**, ghép lại thế nào?". Ví dụ chiều cao, số nút, tổng giá trị:

In [5]:
def chieu_cao(node):
    if node is None:
        return 0                                   # cây rỗng cao 0
    return 1 + max(chieu_cao(node.left), chieu_cao(node.right))

def so_nut(node):
    return 0 if node is None else 1 + so_nut(node.left) + so_nut(node.right)

print("chiều cao:", chieu_cao(cay), "| số nút:", so_nut(cay))

chiều cao: 3 | số nút: 6


### Lỗi hay gặp

- Quên trường hợp `node is None` → `AttributeError: 'NoneType' object has no attribute 'left'`.
- Nhầm "lá" (không có con nào) với "nút thiếu một con".
- Nhầm thứ tự preorder/inorder/postorder.
- Cây lệch (như danh sách) cao tới `n` → đệ quy sâu `n` tầng.

## 2.2 Duyệt theo chiều sâu (DFS — Depth-First Search)

### DFS là gì?

DFS đi **sâu nhất có thể** theo một nhánh, tới khi hết đường thì **quay lui** về ngã rẽ gần nhất và thử nhánh khác — giống đi mê cung bằng cách luôn bám một bức tường.

```text
        A
       / \          DFS từ A (thử hàng xóm theo thứ tự):
      B   C         A → B → D (cụt, quay lui) → E → F → C
     / \   \
    D   E - F
```

Công cụ: **đệ quy** (dùng call stack) hoặc **ngăn xếp** tự quản lý.

### Khung DFS

```text
def dfs(u):
    visited.add(u)
    xử lý u
    for v in hàng_xóm(u):
        if v not in visited:
            dfs(v)
```

In [6]:
do_thi_dfs = {
    "A": ["B", "C"], "B": ["A", "D", "E"], "C": ["A", "F"],
    "D": ["B"], "E": ["B", "F"], "F": ["C", "E"],
}

def dfs_de_quy(graph, u, visited, thu_tu, sau=0):
    visited.add(u)
    thu_tu.append(u)
    print("    " * sau + f"vào {u}")
    for v in graph[u]:
        if v not in visited:
            dfs_de_quy(graph, v, visited, thu_tu, sau + 1)
    print("    " * sau + f"rời {u}")

tt = []
dfs_de_quy(do_thi_dfs, "A", set(), tt)
print("Thứ tự DFS:", tt)

vào A
    vào B
        vào D
        rời D
        vào E
            vào F
                vào C
                rời C
            rời F
        rời E
    rời B
rời A
Thứ tự DFS: ['A', 'B', 'D', 'E', 'F', 'C']


### DFS bằng ngăn xếp (không đệ quy)

Thay hàng đợi của BFS bằng **ngăn xếp**. Tránh được giới hạn độ sâu đệ quy khi dữ liệu lớn.

In [7]:
def dfs_ngan_xep(graph, start):
    st, visited, thu_tu = [start], set(), []
    while st:
        u = st.pop()
        if u in visited:
            continue
        visited.add(u)
        thu_tu.append(u)
        for v in reversed(graph[u]):        # đảo để hàng xóm đầu tiên được lấy ra trước
            if v not in visited:
                st.append(v)
    return thu_tu

print("DFS bằng ngăn xếp:", dfs_ngan_xep(do_thi_dfs, "A"))

DFS bằng ngăn xếp: ['A', 'B', 'D', 'E', 'F', 'C']


### DFS trên lưới: tô loang (flood fill) và đếm vùng

Lưới là đồ thị ẩn: mỗi ô nối với 4 ô kề. DFS từ một ô sẽ "loang" khắp vùng liên thông chứa nó. Đếm số lần phải bắt đầu DFS mới = số vùng.

In [8]:
def dem_dao(grid):
    g = [list(row) for row in grid]
    R, C = len(g), len(g[0])
    def loang(r, c):
        if r < 0 or r >= R or c < 0 or c >= C or g[r][c] != "1":
            return                              # ra ngoài / nước / đã thăm
        g[r][c] = "#"                           # đánh dấu đã thăm
        for dr, dc in [(1, 0), (-1, 0), (0, 1), (0, -1)]:
            loang(r + dr, c + dc)
    dem = 0
    for r in range(R):
        for c in range(C):
            if g[r][c] == "1":
                dem += 1
                loang(r, c)
    return dem, ["".join(row) for row in g]

print(dem_dao(["11000",
               "11010",
               "00011"]))

(2, ['##000', '##0#0', '000##'])


### DFS trên cây: mang thông tin dọc đường đi

Trên cây không cần `visited` (không có chu trình). DFS hay **truyền xuống** thông tin của đường đi từ gốc (tổng, đường đi hiện tại) và **trả lên** kết quả của cây con.

In [9]:
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

cay_dfs = TreeNode(5, TreeNode(4, TreeNode(11, TreeNode(7), TreeNode(2))), TreeNode(8, TreeNode(13), TreeNode(4)))

def moi_duong_goc_la(node, duong, out):
    if node is None:
        return
    duong.append(node.val)                     # đi xuống: thêm nút
    if node.left is None and node.right is None:
        out.append((list(duong), sum(duong)))  # tới lá: ghi lại một bản sao
    moi_duong_goc_la(node.left, duong, out)
    moi_duong_goc_la(node.right, duong, out)
    duong.pop()                                # quay lên: bỏ nút

ket_qua = []
moi_duong_goc_la(cay_dfs, [], ket_qua)
for d, s in ket_qua:
    print(d, "tổng =", s)

[5, 4, 11, 7] tổng = 27
[5, 4, 11, 2] tổng = 22
[5, 8, 13] tổng = 26
[5, 8, 4] tổng = 17


### So sánh BFS và DFS

| | BFS | DFS |
|---|---|---|
| Cấu trúc | hàng đợi | ngăn xếp / đệ quy |
| Thứ tự | theo lớp (gần → xa) | theo nhánh (sâu trước) |
| Đường ngắn nhất (không trọng số) | **có** | không đảm bảo |
| Bộ nhớ | độ rộng lớn nhất | độ sâu lớn nhất |
| Hợp với | khoảng cách, theo tầng | liệt kê đường đi, vùng liên thông, quay lui |

Lỗi hay gặp: quên `visited` trên đồ thị có chu trình (lặp vô hạn); đệ quy quá sâu trên lưới lớn; ghi `out.append(duong)` thay vì bản sao `list(duong)` (về sau list bị sửa).

## 2.3 Áp dụng vào bài này

**DFS** từ gốc, **mang theo số đang ghép** dọc đường đi: đi xuống nút có giá trị `d` thì số mới = `số_cũ * 10 + d`; tới **lá** thì cộng số đó vào tổng.

# 3. Nền tảng

## Ghép chữ số

Đọc số từ trái sang phải: mỗi chữ số mới, nhân số cũ với 10 rồi cộng.

In [10]:
from typing import List, Optional
from collections import deque
import random

cur = 0
for d in [4, 9, 5]:
    cur = cur * 10 + d
    print(d, "->", cur)

4 -> 4
9 -> 49
5 -> 495


In [11]:
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right


def build_tree(values: list) -> Optional[TreeNode]:
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    q = deque([root])
    i = 1
    while q and i < len(values):
        node = q.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i]); q.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i]); q.append(node.right)
        i += 1
    return root

# 4. Baseline: liệt kê đường đi dạng chuỗi rồi cộng

In [12]:
def paths_as_strings(root):
    if not root:
        return []
    if not root.left and not root.right:
        return [str(root.val)]
    return [str(root.val) + p for c in (root.left, root.right) for p in paths_as_strings(c)]

def sum_via_strings(root):
    return sum(int(p) for p in paths_as_strings(root))

print(paths_as_strings(build_tree([4, 9, 0, 5, 1])), sum_via_strings(build_tree([4, 9, 0, 5, 1])))

['495', '491', '40'] 1026


Đúng, nhưng tạo nhiều chuỗi tạm. Ta có thể ghép **số** trực tiếp.

# 5. Ý tưởng chính: DFS mang theo `cur`

```text
dfs(node, cur):
    if node is None: return 0
    cur = cur * 10 + node.val
    if node là lá: return cur
    return dfs(node.left, cur) + dfs(node.right, cur)
```

`cur` là số nguyên → mỗi lời gọi có bản riêng, không cần "hoàn tác" như `path` ở #113.

In [13]:
def sum_numbers_rec(root: Optional[TreeNode]) -> int:
    def dfs(node, cur):
        if not node:
            return 0
        cur = cur * 10 + node.val
        if not node.left and not node.right:
            return cur
        return dfs(node.left, cur) + dfs(node.right, cur)
    return dfs(root, 0)

print(sum_numbers_rec(build_tree([1, 2, 3])), sum_numbers_rec(build_tree([4, 9, 0, 5, 1])))

25 1026


# 6. Mô phỏng

In [14]:
def trace_sum(values):
    root = build_tree(values)
    def dfs(node, cur, depth):
        if not node:
            return 0
        cur = cur * 10 + node.val
        ind = "  " * depth
        if not node.left and not node.right:
            print(f"{ind}lá {node.val}: số = {cur}")
            return cur
        print(f"{ind}nút {node.val}: cur = {cur}")
        s = dfs(node.left, cur, depth + 1) + dfs(node.right, cur, depth + 1)
        print(f"{ind}-> tổng dưới {node.val} = {s}")
        return s
    total = dfs(root, 0, 0)
    print("Kết quả:", total)
    return total

_ = trace_sum([4, 9, 0, 5, 1])

nút 4: cur = 4
  nút 9: cur = 49
    lá 5: số = 495
    lá 1: số = 491
  -> tổng dưới 9 = 986
  lá 0: số = 40
-> tổng dưới 4 = 1026
Kết quả: 1026


# 7. Cách 2: Lặp bằng stack

In [15]:
def sum_numbers_stack(root: Optional[TreeNode]) -> int:
    total, stack = 0, [(root, 0)]
    while stack:
        node, cur = stack.pop()
        if not node:
            continue
        cur = cur * 10 + node.val
        if not node.left and not node.right:
            total += cur
        else:
            stack.append((node.right, cur))
            stack.append((node.left, cur))
    return total

print(sum_numbers_stack(build_tree([1, 2, 3])), sum_numbers_stack(build_tree([4, 9, 0, 5, 1])))

25 1026


Đổi `stack.pop()` thành `deque.popleft()` là được BFS — kết quả như nhau vì phép cộng không phụ thuộc thứ tự.

# 8. Phiên bản đúng format LeetCode

In [16]:
class Solution:
    def sumNumbers(self, root: Optional[TreeNode]) -> int:
        def dfs(node: Optional[TreeNode], cur: int) -> int:
            if not node:
                return 0
            cur = cur * 10 + node.val
            if not node.left and not node.right:
                return cur
            return dfs(node.left, cur) + dfs(node.right, cur)
        return dfs(root, 0)

In [17]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<16} PASS  {actual}")


def random_tree(n):
    if n == 0:
        return None
    k = random.randint(0, n - 1)
    return TreeNode(random.randint(0, 9), random_tree(k), random_tree(n - 1 - k))


sol = Solution()
check("Ví dụ 1", sol.sumNumbers(build_tree([1, 2, 3])), 25)
check("Ví dụ 2", sol.sumNumbers(build_tree([4, 9, 0, 5, 1])), 1026)
check("Một nút", sol.sumNumbers(build_tree([7])), 7)
check("Số 0 ở đầu", sol.sumNumbers(build_tree([0, 1])), 1)
check("Một con", sol.sumNumbers(build_tree([1, 0])), 10)

random.seed(129)
for _ in range(3000):
    root = random_tree(random.randint(1, 15))
    expected = sum_via_strings(root)
    assert sol.sumNumbers(root) == expected == sum_numbers_stack(root)
print("3000 cây ngẫu nhiên: khớp cách ghép chuỗi")

Ví dụ 1          PASS  25
Ví dụ 2          PASS  1026
Một nút          PASS  7
Số 0 ở đầu       PASS  1
Một con          PASS  10
3000 cây ngẫu nhiên: khớp cách ghép chuỗi


# 9. Vì sao đúng? Độ phức tạp?

**Bất biến:** khi xử lý `node`, `cur` (sau khi cập nhật) là số tạo bởi các chữ số từ gốc tới `node`. Ở lá, đó chính là số của đường đi; tổng trả về của mỗi cây con là tổng các số của các lá trong nó.

```text
Time:  O(n)
Space: O(h) cho đệ quy / stack (h ≤ 10 theo đề)
```

# 10. Những lỗi dễ mắc

## Lỗi 1: Cộng `cur` ở nút `None`

In [18]:
def wrong_add_at_none(root):
    def dfs(node, cur):
        if not node:
            return cur                            # SAI: None không phải lá
        cur = cur * 10 + node.val
        return dfs(node.left, cur) + dfs(node.right, cur)
    return dfs(root, 0)

print("Sai :", wrong_add_at_none(build_tree([1, 2, 3])), "| Đúng:", sol.sumNumbers(build_tree([1, 2, 3])))
print("Sai :", wrong_add_at_none(build_tree([1, 0])), "| Đúng:", sol.sumNumbers(build_tree([1, 0])))

Sai :

 50 | Đúng: 25
Sai : 21 | Đúng: 10


Mỗi lá có hai con `None` → mỗi số bị cộng **hai lần**; nút một con còn cộng thêm số "dừng giữa chừng" ở phía `None`.

## Lỗi 2: Cộng chữ số thay vì ghép

`cur + node.val` cho tổng các chữ số, không phải số.

In [19]:
def wrong_digit_sum(root):
    def dfs(node, cur):
        if not node:
            return 0
        cur = cur + node.val                      # SAI
        if not node.left and not node.right:
            return cur
        return dfs(node.left, cur) + dfs(node.right, cur)
    return dfs(root, 0)

print("Sai :", wrong_digit_sum(build_tree([4, 9, 0, 5, 1])), "| Đúng:", sol.sumNumbers(build_tree([4, 9, 0, 5, 1])))

Sai : 36 | Đúng: 1026


## Lỗi 3: Dùng biến toàn cục `cur` mà không hoàn tác

Nếu `cur` là biến dùng chung (không phải tham số), phải `cur //= 10` khi quay lui; quên bước này thì nhánh sau mang theo chữ số của nhánh trước.

## Lỗi 4: Ghép theo thứ tự ngược (`node.val * 10^depth + cur`)

Gốc là chữ số **hàng cao nhất**; ghép `cur * 10 + val` khi đi xuống là đúng thứ tự.

# 11. Mẹo nhận diện pattern

**DFS mang theo trạng thái nhỏ, bất biến** (số đang ghép, tổng đang cộng, độ sâu, min/max trên đường) → truyền qua tham số, không cần backtracking.

Bài liên quan: **#112 / #113 Path Sum**, **#257 Binary Tree Paths**, **#988 Smallest String Starting From Leaf**, **#1022 Sum of Root To Leaf Binary Numbers** (ghép bit: `cur * 2 + val`), **#1448 Count Good Nodes**.

# 12. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Cây `[3, 1, 4, None, 5, 9, 2]`: liệt kê các số và tính tổng.

## Bài 2 — Điền chỗ trống

In [20]:
def exercise_2_sum_numbers(root: Optional[TreeNode]) -> int:
    def dfs(node, cur):
        # TODO
        pass
    return dfs(root, 0)


# assert exercise_2_sum_numbers(build_tree([1, 2, 3])) == 25
# assert exercise_2_sum_numbers(build_tree([4, 9, 0, 5, 1])) == 1026

## Bài 3 — Biến thể: Sum of Root To Leaf Binary Numbers (LeetCode #1022)

Mỗi nút là bit `0/1`; mỗi đường là một số **nhị phân**.

```python
sum_root_to_leaf_binary([1, 0, 1, 0, 1, 0, 1]) -> 22     # 100 + 101 + 110 + 111 = 4 + 5 + 6 + 7
```

In [21]:
def exercise_3_sum_binary(root: Optional[TreeNode]) -> int:
    # TODO
    pass


# assert exercise_3_sum_binary(build_tree([1, 0, 1, 0, 1, 0, 1])) == 22
# assert exercise_3_sum_binary(build_tree([0])) == 0

## Bài 4 — Biến thể: Count Good Nodes in Binary Tree (LeetCode #1448)

Nút `X` là "tốt" nếu trên đường từ gốc tới `X` không có nút nào **lớn hơn** `X`. Đếm số nút tốt.

```python
good_nodes([3, 1, 4, 3, None, 1, 5]) -> 4
good_nodes([3, 3, None, 4, 2])       -> 3
```

In [22]:
def exercise_4_good_nodes(root: Optional[TreeNode]) -> int:
    # TODO
    pass


# assert exercise_4_good_nodes(build_tree([3, 1, 4, 3, None, 1, 5])) == 4
# assert exercise_4_good_nodes(build_tree([3, 3, None, 4, 2])) == 3
# assert exercise_4_good_nodes(build_tree([1])) == 1

## Bài 5 — Mini challenge: Smallest String Starting From Leaf (LeetCode #988)

Mỗi nút là số `0..25` ứng với `'a'..'z'`. Tìm chuỗi **nhỏ nhất theo từ điển** đọc **từ lá lên gốc**.

```python
smallest_from_leaf([0, 1, 2, 3, 4, 3, 4])    -> "dba"
smallest_from_leaf([25, 1, 3, 1, 3, 0, 2])   -> "adz"
```

Gợi ý nhanh: mang theo chuỗi từ gốc; ở lá đảo ngược rồi so sánh. Chú ý: **không** được chọn tham lam ở từng nút — so sánh tiền tố ngắn có thể đánh lừa.

In [23]:
def exercise_5_smallest_from_leaf(root: Optional[TreeNode]) -> str:
    # TODO
    pass


# assert exercise_5_smallest_from_leaf(build_tree([0, 1, 2, 3, 4, 3, 4])) == "dba"
# assert exercise_5_smallest_from_leaf(build_tree([25, 1, 3, 1, 3, 0, 2])) == "adz"
# assert exercise_5_smallest_from_leaf(build_tree([2, 2, 1, None, 1, 0, None, 0])) == "abc"

# 13. Gợi ý

**Bài 1**
- Mức 1: chạy `trace_sum` sau khi tự làm.

**Bài 3**
- Mức 1: như bài chính với `cur * 2 + val` (hoặc `cur << 1 | val`).

**Bài 4**
- Mức 1: mang theo `max_so_far`; nút tốt nếu `val >= max_so_far`.

**Bài 5**
- Mức 1: `dfs(node, s)` với `s = chr(97 + val) + s` (ghép **phía trước** → đã là thứ tự lá lên gốc).
- Mức 2: ở lá, `best = min(best, s)`.

# 14. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [24]:
_ = trace_sum([3, 1, 4, None, 5, 9, 2])

nút 3: cur = 3
  nút 1: cur = 31
    lá 5: số = 315
  -> tổng dưới 1 = 315
  nút 4: cur = 34
    lá 9: số = 349
    lá 2: số = 342
  -> tổng dưới 4 = 691
-> tổng dưới 3 = 1006
Kết quả: 1006


## Đáp án Bài 2

In [25]:
def answer_2_sum_numbers(root: Optional[TreeNode]) -> int:
    return sum_numbers_rec(root)

assert answer_2_sum_numbers(build_tree([1, 2, 3])) == 25
assert answer_2_sum_numbers(build_tree([4, 9, 0, 5, 1])) == 1026
print("Bài 2 PASS (giống sum_numbers_rec ở mục 5)")

Bài 2 PASS (giống sum_numbers_rec ở mục 5)


## Đáp án Bài 3

In [26]:
def answer_3_sum_binary(root: Optional[TreeNode]) -> int:
    def dfs(node, cur):
        if not node:
            return 0
        cur = cur * 2 + node.val
        if not node.left and not node.right:
            return cur
        return dfs(node.left, cur) + dfs(node.right, cur)
    return dfs(root, 0)

assert answer_3_sum_binary(build_tree([1, 0, 1, 0, 1, 0, 1])) == 22
assert answer_3_sum_binary(build_tree([0])) == 0
print("Bài 3 PASS")

Bài 3 PASS


## Đáp án Bài 4

In [27]:
def answer_4_good_nodes(root: Optional[TreeNode]) -> int:
    def dfs(node, mx):
        if not node:
            return 0
        good = node.val >= mx
        mx = max(mx, node.val)
        return good + dfs(node.left, mx) + dfs(node.right, mx)
    return dfs(root, float("-inf"))

assert answer_4_good_nodes(build_tree([3, 1, 4, 3, None, 1, 5])) == 4
assert answer_4_good_nodes(build_tree([3, 3, None, 4, 2])) == 3
assert answer_4_good_nodes(build_tree([1])) == 1
print("Bài 4 PASS")

Bài 4 PASS


## Đáp án Bài 5

In [28]:
def answer_5_smallest_from_leaf(root: Optional[TreeNode]) -> str:
    best = None
    def dfs(node, s):
        nonlocal best
        if not node:
            return
        s = chr(97 + node.val) + s
        if not node.left and not node.right:
            if best is None or s < best:
                best = s
            return
        dfs(node.left, s)
        dfs(node.right, s)
    dfs(root, "")
    return best

assert answer_5_smallest_from_leaf(build_tree([0, 1, 2, 3, 4, 3, 4])) == "dba"
assert answer_5_smallest_from_leaf(build_tree([25, 1, 3, 1, 3, 0, 2])) == "adz"
assert answer_5_smallest_from_leaf(build_tree([2, 2, 1, None, 1, 0, None, 0])) == "abc"
print("Bài 5 PASS")

Bài 5 PASS


# 15. Tổng kết

> DFS mang theo `cur = cur * 10 + node.val`; ở **lá** trả `cur`, ở nút trong trả tổng của hai con, ở `None` trả 0. `O(n)` thời gian, `O(h)` bộ nhớ.

---

## Checklist trước khi nộp

- [ ] Ghép `cur * 10 + val` (không cộng chữ số)?
- [ ] Chỉ cộng ở **lá** (`not left and not right`)?
- [ ] `None` trả 0?
- [ ] Cây một con xử lý đúng (như `[1, 0]` → 10)?